# Explore FPL data
Run `make data` first (or drop the two parquet files into `data/interim/` and `data/processed/`).

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 60)
pg = pd.read_parquet('../data/interim/player_gw.parquet')   # one row per player x fixture
f = pd.read_parquet('../data/processed/features.parquet')   # one row per player x GW, features + targets
pg.shape, f.shape

In [ ]:
# rows per season, share of players who played
pg.groupby('season').agg(rows=('player_id','size'), players=('player_id','nunique'), played=('minutes', lambda m: (m>0).mean()))

In [ ]:
# one player across seasons (IDs are stable `code`s, names are display only)
pg[pg.web_name.str.contains('Salah')].groupby('season')[['total_points','minutes','expected_goals']].sum()

In [ ]:
# role vs points
real = pg[~pg.is_ghost_fixture & (pg.position != 'AM')]
real.groupby(['role'])['total_points'].describe()

In [ ]:
# live season: next GW (11) feature rows
live = f[(f.season == '2026-27') & (f.gw == 11)]
live.sort_values('pts_r5_blend', ascending=False)[['web_name','position','price','fdr','pts_r5_blend','minutes_r5','role','status_asof','fpl_xp']].head(20)

In [ ]:
d = f[f.y_1.notna() & (f.season >= '2020-21')]
num = d.select_dtypes('number').columns.drop(['y_1','y_3','y_5','total_points','minutes'], errors='ignore')
d[num].corrwith(d.y_1, method='spearman').sort_values(ascending=False).head(25)